# Notebook initialization

In [ ]:
import pyxdf
import numpy as np
import os
import logging
from PIL import Image


# # for the tests
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.widgets import SpanSelector

from scipy.signal import butter, sosfiltfilt, find_peaks


# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True

if doRunTests:
    import matplotlib.pyplot as plt
    # the best for debug-test plots (external window that you can make fullscreen and zoom)
    %matplotlib qt


    # functions for plotting the tests... 
    def plot_before_after(time_before, data_before, time_after, data_after, title_txt=""):
        """
        Plot the data before and after doing some changes.
        """
        plt.figure()
        plt.plot(time_before, data_before, "+-", label="before")
        plt.plot(time_after, data_after, "*", label="after")
        plt.title(f"Data: before and after {title_txt}")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.show()

# Classes to manage XDF files

In [ ]:
class XDF_file:
    """
    Class to handle XDF files.
    """

    def __init__(
        self,
        xdf_fullFname: str | os.PathLike,
        select_streams=None,
    ):
        """
        Initialize the XDF file.
        Parameters
        ----------
        xdf_fullFname : str
            Full path to the XDF file.
        select_streams : list of str (as in load_xdf)
            List of stream types to select. If None, all streams are selected.
        """
        self.xdf_fullFname = xdf_fullFname
        self.xdf_fname = os.path.basename(xdf_fullFname)
        self.xdf_dir = os.path.dirname(xdf_fullFname)
        xdf_streams, xdf_header = pyxdf.load_xdf(
            xdf_fullFname,
            select_streams=select_streams,
            synchronize_clocks=True,
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
            verbose=False,
        )

        self.streams = []
        for i in range(len(xdf_streams)):
            stream = xdf_streams[i]
            self.streams.append(XDF_stream(stream))

    def __str__(self):
        """Print the names and types of all streams in the xdf file"""
        s = ""
        for i in range(len(self.streams)):
            stream = self.streams[i]
            s_type = stream.type
            s_name = stream.name
            s += f"Stream {i}: {s_type}, {s_name}\n"
        return s

    def get_stream_index(self, searched_stream_type, searched_stream_names):
        """Get the index of the stream of type 'searched_stream_type' AND with name in
        'searched_stream_names'"""

        if not isinstance(searched_stream_names, list):
            # if we get a string (only one name)
            searched_stream_names = [searched_stream_names]

        found_streams = []
        for i, stream in enumerate(self.streams):
            if searched_stream_type == stream.type:
                for searched_stream_name in searched_stream_names:
                    if searched_stream_name == stream.name:
                        found_streams.append(i)

        if not found_streams:
            return None

        if len(found_streams) > 1:
            found_streams_names = [self.streams[i].name for i in found_streams]
            msg = (
                f"Found multiple streams: "
                f"[{searched_stream_type},{found_streams_names}]."
            )
            raise ValueError(msg)

        return found_streams[0]


class XDF_channel:
    """
    Class to handle XDF channels.
    """

    def __init__(self, index, stream):
        self.index = index
        desc = stream["info"]["desc"][0]["channels"][0]["channel"][index]
        self.label = desc["label"][0]
        self.type = desc["type"][0]
        self.unit = desc["unit"][0]

        self.time_series = stream["time_series"][:, index]
        self.time_stamps = stream["time_stamps"]

    def __str__(self):
        """Print the name and type of the channel"""
        s = f"Channel {self.index}: {self.label} ({self.type},  {self.unit})\n"
        return s


class XDF_stream:
    """
    Class to handle XDF streams.
    A stream can be organized by channel (stream.channels = [...],  e.g. for data) or
    not organized (stream.channels = [], e.g. for markers ).
    """

    def __init__(self, xdf_stream):
        self.xdf_stream = xdf_stream
        self.time_stamps = xdf_stream["time_stamps"]
        self.time_series = xdf_stream["time_series"]
        self.name = xdf_stream["info"]["name"][0]
        self.type = xdf_stream["info"]["type"][0]
        self.channels = self.set_channels()

    def __str__(self):
        """Print the names and types of all channels in the stream"""
        s = f"Stream {self.name} ({self.type})\n"
        for i in range(len(self.channels)):
            channel_name = self.channels[i].label
            channel_type = self.channels[i].type
            channel_unit = self.channels[i].unit
            s += f"Channel {i}: {channel_name} ({channel_type}, {channel_unit})\n"
        return s

    def set_channels(self):
        """Set the channels from the xdf stream"""
        channels = []
        try:
            n_channels = len(
                self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"]
            )
            for i in range(n_channels):
                channel = XDF_channel(i, self.xdf_stream)
                channels.append(channel)
        except Exception:
            pass

        return channels

    def get_one_channel(self, name):
        for i in range(len(self.channels)):
            channel = self.channels[i]
            if channel.label == name:
                return channel
        return None

    def get_channel_index(self, channel_name):
        """Get the index of one channel from the stream by its name"""
        channel_index = -1
        nb_channels = len(self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"])
        for i in range(nb_channels):
            current_name = self.xdf_stream["info"]["desc"][0]["channels"][0]["channel"][
                i
            ]["label"][0]
            if current_name == channel_name:
                channel_index = i
                break
        if channel_index == -1:
            return None

        return channel_index


def interpolate_to_constant_time_step(t, x, dt=1 / 30):
    """Interpolate the data to a constant time step"""

    n_columns = x.shape[1] if x.ndim > 1 else 1

    t_new = np.arange(t[0], t[-1], dt)

    if x.ndim < 2:
        x_new = np.interp(t_new, t, x)
    else:
        x_new = np.zeros((len(t_new), n_columns))
        for i in range(n_columns):
            x_new[:, i] = np.interp(t_new, t, x[:, i])

    return x_new, t_new

In [ ]:
if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"  # no mouse marker csv file associated -> nan time correction - solution found
    xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231009_V1/ReArm_C1P38_20231009_V1_Reaching/ReArm_C1P38_20231009_V1_r.xdf"  # Test file for panu identification

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    print(xdf_file)
    i_kinect = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_kinect:
        print(xdf_file.streams[i_kinect])  # print the stream = all channels
        print(xdf_file.streams[i_kinect].channels[0])
        print(xdf_file.streams[i_kinect].get_one_channel("SpineBase_X"))

## Make the kinect time correction (if needed)

Any `*.xdf` file needing a correction of the kinect timestamps is accompanied by a `*_xdf_time_correction.csv` file with the time correction. 
If no such a file is present, no correction is needed.

In [ ]:
def get_time_correction(xdf_file: XDF_file):
    """Get the time correction from the xdf file name"""

    xdf_fullFname = str(xdf_file.xdf_fullFname)
    is_log_present = False
    log_fullFname = os.path.join(
        os.path.dirname(xdf_fullFname), "..", "kinect_time_correction.log"
    )
    log_fullFname = os.path.normpath(log_fullFname)
    if os.path.exists(log_fullFname):
        is_log_present = True

    is_correction_present = False
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    if os.path.exists(time_correction_file):
        is_correction_present = True

    if not is_log_present:
        raise FileNotFoundError(
            f"The file {log_fullFname} does not exist. Please run the script to generate it."
        )

    if not is_correction_present:
        time_correction = np.float64(0)
    else:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)

    return time_correction


def make_kinect_time_correction(xdf_file: XDF_file):
    """Make the time correction for the Kinect streams in the xdf file."""

    # get the time correction from the xdf file name
    time_correction = get_time_correction(xdf_file)

    if time_correction == 0:
        # no time correction needed
        return

    if np.isnan(time_correction):
        # this can happen when the mouse csv file is not present...
        logging.warning(
            f"Time correction is NaN. Is there a mouse csv file for {xdf_file.xdf_fullFname}?"
        )
        return

    # here, we are OK to make the time correction
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    i_k_mk = xdf_file.get_stream_index("Markers", "EuroMov-Markers-Kinect")

    if i_k_mo is None or i_k_mk is None:
        raise ValueError(
            f"Cannot find the Kinect streams in {xdf_file.xdf_fullFname}. "
            f"Please check the stream names."
        )

    xdf_file.streams[i_k_mo].time_stamps += time_correction
    xdf_file.streams[i_k_mk].time_stamps += time_correction
    logging.info(
        f"Time correction of {time_correction} s applied to the Kinect streams."
    )

# Remove all rows filled with only zeros in the kinect data

In [ ]:
def remove_zero_rows(xdf_file: XDF_file):
    """
    Remove the rows filled with zeros from the kinect data.
    """

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        kinect_t = xdf_file.streams[i_k_mo].time_stamps
        kinect_data = xdf_file.streams[i_k_mo].time_series

        # find the indexes of kinect data that are filled with zeros
        zero_rows = np.all(kinect_data == 0, axis=1)
        zero_rows_indices = np.where(zero_rows)[0]

        if len(zero_rows_indices) > 0:
            # remove the zero rows from the data
            kinect_data = np.delete(kinect_data, zero_rows_indices, axis=0)
            kinect_t = np.delete(kinect_t, zero_rows_indices, axis=0)

            # modify the stream (but not the original xdf_file.xdf_streams )
            xdf_file.streams[i_k_mo].time_series = kinect_data
            xdf_file.streams[i_k_mo].time_stamps = kinect_t

            logging.info(
                f"Removed {len(zero_rows_indices)} rows filled with only zeros in the Kinect data"
            )


if doRunTests:

    # test the function
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo is None:
        raise ValueError("No MoCap stream found")

    i_wz = xdf_file.streams[i_k_mo].get_channel_index("WristRight_Z")

    t_before = xdf_file.streams[i_k_mo].time_stamps.copy()
    data_before = xdf_file.streams[i_k_mo].time_series[:, i_wz].copy()

    remove_zero_rows(xdf_file)

    t_after = xdf_file.streams[i_k_mo].time_stamps
    data_after = xdf_file.streams[i_k_mo].time_series[:, i_wz]

    plot_before_after(t_before, data_before, t_after, data_after)


# Interpolate the Mocap data
This is mandatory because the kinect and mouse data are produced by the computer: the sampling rate is not waranted to be constant (samples are forgetten... sometimes). 

In [ ]:
def resample_kinect_data(xdf_file: XDF_file):
    """
    Resample the Kinect data to a constant time step.
    """

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo:
        kinect_t = xdf_file.streams[i_k_mo].time_stamps
        kinect_data = xdf_file.streams[i_k_mo].time_series

        initial_sampling_rate = 1 / (kinect_t[1] - kinect_t[0])

        # resample the data to a constant time step
        time_step = 1 / 30
        kinect_data, kinect_t = interpolate_to_constant_time_step(
            kinect_t, kinect_data, dt=time_step
        )

        # modify the stream (but not the original xdf_file.xdf_streams )
        xdf_file.streams[i_k_mo].time_series = kinect_data
        xdf_file.streams[i_k_mo].time_stamps = kinect_t

        logging.info(
            f"Kinect resampled at {1/time_step:3.2f} Hz (from about {initial_sampling_rate:3.2f} Hz before)."
        )


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution
    # NOTE: the sampling rate is 15hz in this xdf file

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        remove_zero_rows(xdf_file)

        stream = xdf_file.streams[i_k_mo]
        i_wz = stream.get_channel_index("WristRight_Z")
        if i_wz is None:
            raise ValueError("No WristRight_Z channel found")
        t_before = stream.time_stamps.copy()
        data_before = stream.time_series[:, i_wz].copy()

        resample_kinect_data(xdf_file)

        t_after = stream.time_stamps
        data_after = stream.time_series[:, i_wz]
        plot_before_after(t_before, data_before, t_after, data_after)

# Low pass filter 

In [ ]:
def butter_lowpass(cutoff, fs, order=2):
    """Design a lowpass Butterworth filter."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    sos = butter(
        order,
        normal_cutoff,
        btype="low",
        output="sos",  # recommended for numerical stability by scipy
    )
    return sos


def lowpass_filter(t, data, cutoff=0.5):
    """Apply a lowpass filter to the data"""
    # check that the sampling period is constant
    dt = np.mean(np.diff(t))
    if not np.allclose(np.diff(t), dt):
        raise ValueError("The time vector do not have a constant sampling period")

    fs = 1 / dt  # sample rate, Hz
    order = 4  # order of the filter
    sos = butter_lowpass(cutoff, fs, order=order)
    filtered_data = sosfiltfilt(sos, data)
    return filtered_data


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros in the beginning -> incomplete data - no solution

    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")

    if i_k_mo:
        remove_zero_rows(xdf_file)
        resample_kinect_data(xdf_file)

        stream = xdf_file.streams[i_k_mo]
        i_wz = stream.get_channel_index("WristRight_Z")
        if i_wz is None:
            raise ValueError("No WristRight_Z channel found")
        t_before = stream.time_stamps.copy()
        data_before = stream.time_series[:, i_wz].copy()

        filtered_data = lowpass_filter(t_before, data_before, cutoff=0.5)

        plot_before_after(t_before, data_before, t_before, filtered_data, "filter")

# Get the action zones from the markers
The participant is asked to follow a timed sequence: 
- Action: perform a series of reaches (20s)
- Pause: wait for the next action (20s)
- ...
- Action: perform a series of reaches (20s)
- Pause: wait for the next action (20s)


NB: Outside action zones, the data is not relevant for the reaches analysis, *and it adds a lot of noise to the data*.

The markers are necessary to identify the action zones, within which the reaches (are expected to) occur. 

Each action zone has a *start* and *end* marker, but the label of the markers differs if the sequence was generated by : 

- the software **LSL-Mouse**: stream `mouse_to_nic_markers`
    - start = `"[111]"`  
    - stop = `"[100]"` 

- the software **event-IDE**: stream `event_to_nic_markers`
    - start = `"[100]"`, but we have to keep only the first start in the sequence before each stop
    - stop = `"[75]"`


The streams are loaded from the xdf file using the function `get_stream`:
``` python
# get the markers streams from the xdf file
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])
event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"]) 
```
``` python

In [ ]:
def find_marker_indexes(marker_name: str, markers_data: list):
    """Find the indexes of marker_name in markers_data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return np.array(marker_index_list)


def get_coherent_actions_start_stop_times(starts, stops):
    """Get the coherent start and stop times of the actions"""

    # NOTE: we need this because of bugs in the data acquisition...

    # make an array of start, 0 and stop, 1
    start = np.zeros(
        len(starts),
        dtype=[("time", float), ("type", int)],
    )
    start["time"] = starts
    start["type"] = 0
    stop = np.zeros(
        len(stops),
        dtype=[("time", float), ("type", int)],
    )
    stop["time"] = stops
    stop["type"] = 1
    start_and_stop = np.concatenate((start, stop))
    start_and_stop = np.sort(start_and_stop, order="time")

    for i in range(len(start_and_stop) - 1):
        # keep only the last start in case of multiple contiguous start
        if start_and_stop[i]["type"] == 0 and start_and_stop[i + 1]["type"] == 0:
            start_and_stop[i + 1]["type"] = -1
        # keep only the first stop in case of multiple contiguous stop
        if start_and_stop[i]["type"] == 1 and start_and_stop[i + 1]["type"] == 1:
            start_and_stop[i + 1]["type"] = -1

    # ensure that the first is a start and the last is a stop
    if start_and_stop[0]["type"] == 1:
        start_and_stop[0]["type"] = -1
    if start_and_stop[-1]["type"] == 0:
        start_and_stop[-1]["type"] = -1

    # clean the start_and_stop array
    start_and_stop_ok = start_and_stop[start_and_stop["type"] != -1]

    starts_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 0]
    stops_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 1]

    if len(starts_corrected) != len(stops_corrected):
        raise ValueError(
            f"Number of starts ({len(starts_corrected)}) and stops ({len(stops_corrected)}) are not equal"
        )

    return starts_corrected, stops_corrected


def get_actions_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop times from mouse_to_nic_markers"""

    mouse_to_nic_markers_data = mouse_to_nic_markers.time_series
    mouse_to_nic_markers_time = mouse_to_nic_markers.time_stamps

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find_marker_indexes("[111]", mouse_to_nic_markers_data)
    stop_marker_index_list = find_marker_indexes("[100]", mouse_to_nic_markers_data)
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    start_times_from_mouse_to_nic_markers, stop_times_from_mouse_to_nic_markers = (
        get_coherent_actions_start_stop_times(
            start_times_from_mouse_to_nic_markers,
            stop_times_from_mouse_to_nic_markers,
        )
    )

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_actions_start_stop_times_from_event_ide_TONIC(event_ide_tonic):
    """get the start and stop times from event_ide_tonic"""

    event_ide_markers_data = event_ide_tonic.time_series
    event_ide_markers_time = event_ide_tonic.time_stamps

    if not isinstance(event_ide_markers_data[0], list):
        event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

    start_marker_index_list = find_marker_indexes("[100]", event_ide_markers_data)
    stop_marker_index_list = find_marker_indexes("[75]", event_ide_markers_data)

    start_times_from_event_ide_tonic = event_ide_markers_time[start_marker_index_list]
    stop_times_from_event_ide_tonic = event_ide_markers_time[stop_marker_index_list]

    # keep only the first start time for each stop time
    previous_stop_time = 0
    good_start_times = []
    for stop_time in stop_times_from_event_ide_tonic:
        possible_start_times = start_times_from_event_ide_tonic[
            start_times_from_event_ide_tonic < stop_time
        ]
        possible_start_times = possible_start_times[
            possible_start_times > previous_stop_time
        ]
        start_time = possible_start_times[0] if len(possible_start_times) > 0 else None
        good_start_times.append(start_time)
        previous_stop_time = stop_time
    good_start_times = np.array(good_start_times)
    good_start_times = good_start_times[
        good_start_times != None  # noqa: E711
    ]  # should be useless...

    good_start_times, stop_times_from_event_ide_tonic = (
        get_coherent_actions_start_stop_times(
            good_start_times,
            stop_times_from_event_ide_tonic,
        )
    )

    return (
        good_start_times,
        stop_times_from_event_ide_tonic,
    )


def get_actions_from_markers(xdf_file: XDF_file):
    """Get the start and stop times of the actions from the markers"""

    i_e2n = xdf_file.get_stream_index("Markers", ["event_ide_TONIC"])
    i_m2n = xdf_file.get_stream_index("Markers", ["MouseToNIC"])

    if i_e2n:
        start_t, stop_t = get_actions_start_stop_times_from_event_ide_TONIC(
            xdf_file.streams[i_e2n]
        )
        stream_name = xdf_file.streams[i_e2n].name
    elif i_m2n:
        start_t, stop_t = get_actions_start_stop_times_from_mouse_to_nic_markers(
            xdf_file.streams[i_m2n]
        )
        stream_name = xdf_file.streams[i_m2n].name
    else:
        raise ValueError("No event_ide_TONIC or MouseToNIC stream found")

    return {
        "start_times": start_t,
        "stop_times": stop_t,
        "stream_name": stream_name,
    }


def print_actions_start_stop_times(start_stop_times):
    """Print the start and stop times of the actions"""

    start_times = start_stop_times["start_times"]
    stop_times = start_stop_times["stop_times"]

    print(f"Stream {start_stop_times['stream_name']}: ")
    for i in range(len(start_times)):
        print(
            f"action{i:02d}: {start_times[i]:8.2f} -> {stop_times[i]:8.2f}, Duration: {stop_times[i] - start_times[i]:5.2f}s"
        )


if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P05/ReArm_C1P05_20210514_V1/ReArm_C1P05_20210514_V1_Reaching/ReArm_C1P05_20210621_V1_r.xdf"  # Test file for panu identification
    # xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231009_V1/ReArm_C1P38_20231009_V1_Reaching/ReArm_C1P38_20231009_V1_r.xdf"  # Test file for panu identification
    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    print(xdf_file)
    actions = get_actions_from_markers(xdf_file)
    print_actions_start_stop_times(actions)

# Get the reaches

In [ ]:
def get_reaches(t, wrist, wrist_f):
    """get the reaches on this wrist"""

    peaks_threshold = np.median(wrist_f) - 0.1
    i_inter_peaks = 60  # 2 seconds

    ########################################################################################
    # alternative way to find the reaches from the peaks (simpler logic)

    # find the negative peaks in the wrist that prominent of 0.1 and at least 2 seconds apart
    # NOTE: prominence works very well but it is difficult to understand the meaning of the value
    # neg_peaks, _ = find_peaks(-wrist_f, prominence=0.1, distance=60)  # 60 = 2s

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    # NOTE: this is the simplest method to find the peaks (same as method 1)
    neg_peaks, _ = find_peaks(-wrist_f, height=-peaks_threshold, distance=i_inter_peaks)

    # remove the peaks that are outliers (all reaches should end at the same target position)
    # NOTE: this is the simplest method (same as method 1)
    reaches_end = wrist_f[neg_peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    neg_peaks = np.delete(neg_peaks, i_outliers)

    # equivalent to index_of_last_negative_velocity_before_peak()... but simpler conceptually
    pos_peaks, _ = find_peaks(wrist_f)

    # put negative and positive peaks in the same list sorted by time
    negative_peaks = [
        {"index": neg_peaks[i], "from": "neg"} for i in range(len(neg_peaks))
    ]
    positive_peaks = [
        {"index": pos_peaks[i], "from": "pos"} for i in range(len(pos_peaks))
    ]
    pks = negative_peaks + positive_peaks
    pks = sorted(pks, key=lambda x: x["index"])

    # for each negative peak, keep only the previous positive peak
    reaches = []
    for i in range(1, len(pks)):
        if pks[i]["from"] == "neg":
            i_end = pks[i]["index"]
            i_beg = pks[i - 1]["index"]
            reach_distance = wrist_f[i_end] - wrist_f[i_beg]
            if -reach_distance > 0.05:  # 5 cm
                reaches.append(
                    {
                        "i_beg": i_beg,
                        "i_end": i_end,
                        "length": reach_distance,
                    }
                )

    # # remove the peaks that are large outliers in distance
    # # NOTE: this makes a difference with the previous method (assert does not pass)
    # # but it is best to remove the outliers in distance (they are outlier!)
    reaches_length = np.array([r["length"] for r in reaches])
    length_median = np.median(reaches_length)
    length_iqr = np.percentile(reaches_length, 75) - np.percentile(reaches_length, 25)
    i_outliers = [
        i
        for i in range(len(reaches_length))
        if abs(reaches_length[i] - length_median) > 3 * length_iqr
    ]
    reaches = np.delete(reaches, i_outliers)

    return neg_peaks, reaches, peaks_threshold


def plot_reaches(
    ax, t, wrist, wrist_f, i_peaks, reaches, label="wrist", color="b", thresh=None
):
    """ " Plot the reaches of the wrist"""
    ax.plot(t, wrist, ".", label=label, color=color)
    ax.plot(t, wrist_f, label=f"{label} filtered", color=color, alpha=0.2)

    # plot the threshold
    if thresh is not None:
        ax.axhline(
            y=thresh,
            color=color,
            linestyle="--",
            label=f"Threshold {label}",
        )
    for reach in reaches:
        # plot the start and end of the reach + a line
        ax.plot(
            t[reach["i_beg"]],
            wrist_f[reach["i_beg"]],
            "o",
            color="orange",
        )
        ax.plot(
            t[reach["i_end"]],
            wrist_f[reach["i_end"]],
            "o",
            color="r",
        )
        ax.plot(
            [t[reach["i_beg"]], t[reach["i_end"]]],
            [wrist_f[reach["i_beg"]], wrist_f[reach["i_end"]]],
            color="k",
            linestyle="--",
        )
        # plot the start and end of the reach on the filtered position
        ax.plot(
            t[reach["i_beg"]],
            wrist_f[t == t[reach["i_beg"]]],
            "x",
            color="orange",
        )
        ax.plot(
            t[reach["i_end"]],
            wrist_f[t == t[reach["i_end"]]],
            "x",
            color="r",
        )

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()


def is_reach_file(xdf_fullFname: str | os.PathLike):
    """Check if the xdf file is a reach file"""

    full_fname = str(xdf_fullFname)
    fname = os.path.basename(full_fname)
    if "Reaching" in full_fname:
        return True

    if "_r.xdf" in fname:
        return True

    if "_Reach" in fname:
        return True

    return False


def get_reaches_from_both_wrist(xdf_file: XDF_file):
    """Get the reaches from both wrists"""

    if not is_reach_file(xdf_file.xdf_fullFname):
        return None

    i_k_mo = xdf_file.get_stream_index("MoCap", "EuroMov-Mocap-Kinect")
    if i_k_mo:
        stream = xdf_file.streams[i_k_mo]
        i_wrz = stream.get_channel_index("WristRight_Z")
        i_wlz = stream.get_channel_index("WristLeft_Z")

        kinect_t = stream.time_stamps

        WristLeft_xyz = stream.time_series[:, i_wlz : i_wlz + 3]
        WristRight_xyz = stream.time_series[:, i_wrz : i_wrz + 3]

        WristLeft_Norm = np.linalg.norm(WristLeft_xyz, axis=1)
        WristRight_Norm = np.linalg.norm(WristRight_xyz, axis=1)

        WristLeft_Norm_f = lowpass_filter(kinect_t, WristLeft_Norm, cutoff=0.5)
        WristRight_Norm_f = lowpass_filter(kinect_t, WristRight_Norm, cutoff=0.5)

        peaks_left, reaches_left, thresh_left = get_reaches(
            kinect_t, WristLeft_Norm, WristLeft_Norm_f
        )
        peaks_right, reaches_right, thresh_right = get_reaches(
            kinect_t, WristRight_Norm, WristRight_Norm_f
        )

        if doRunTests:
            fig, ax = plt.subplots(figsize=(10, 5))
            plot_reaches(
                ax,
                kinect_t,
                WristLeft_Norm,
                WristLeft_Norm_f,
                peaks_left,
                reaches_left,
                label="Left wrist",
                color="b",
                thresh=thresh_left,
            )
            plot_reaches(
                ax,
                kinect_t,
                WristRight_Norm,
                WristRight_Norm_f,
                peaks_right,
                reaches_right,
                label="Right wrist",
                color="k",
                thresh=thresh_right,
            )

            plt.title(
                "Reaches detected from the wrist position (low pass filtered @ 0.5Hz)"
                + f"\n{xdf_file.xdf_fullFname}"
            )
            plt.show()

    logging.info(
        f"Found {len(reaches_left)} reaches on the left wrist and {len(reaches_right)} reaches on the right wrist"
    )
    return {
        "reaches_left": reaches_left,
        "reaches_right": reaches_right,
    }


#########################################################################################

if doRunTests:
    xdf_fullFname = "../dat/ReArm.lnk/PANU/ReArm_C1P38/ReArm_C1P38_20231009_V1/ReArm_C1P38_20231009_V1_Reaching/ReArm_C1P38_20231009_V1_r.xdf"  # Test file for panu identification
    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)

    reaches = get_reaches_from_both_wrist(xdf_file)

    # print the reaches
    print("Left wrist reaches:")
    for i in range(len(reaches["reaches_left"])):
        print(
            f"Reach {i:2d}: {reaches['reaches_left'][i]['length']:.2f} m, "
            f"from {reaches['reaches_left'][i]['i_beg']} to {reaches['reaches_left'][i]['i_end']}"
        )
    print("Right wrist reaches:")
    for i in range(len(reaches["reaches_right"])):
        print(
            f"Reach {i:2d}: {reaches['reaches_right'][i]['length']:.2f} m, "
            f"from {reaches['reaches_right'][i]['i_beg']} to {reaches['reaches_right'][i]['i_end']}"
        )

# Compute panu for one xdf file


In [ ]:
def save_panu(xdf_fullFname):
    fname_xdf = os.path.basename(xdf_fullFname)
    fname_panu = fname_xdf.replace(".xdf", "_xdf_panu.csv")

    # load the xdf file
    xdf_file = XDF_file(
        xdf_fullFname,
        select_streams=[
            {"type": "MoCap"},
            {"type": "Markers"},
        ],
    )

    make_kinect_time_correction(xdf_file)
    remove_zero_rows(xdf_file)
    resample_kinect_data(xdf_file)
    reaches = get_reaches_from_both_wrist(xdf_file)
    actions = get_actions_from_markers(xdf_file)

    logging.info(f"Saved '{fname_panu}'")

# Compute panu for one visit

In [ ]:
def get_xdf_files_in_visit(visit_dir, directories_to_skip=None):
    """Get the xdf files in the visit_dir"""

    xdf_files = []

    if not os.path.exists(visit_dir):
        raise ValueError(f"Directory {visit_dir} does not exist")

    for root, dirs, files in os.walk(visit_dir):
        # Skip the directories that are in the directories_to_skip list
        if directories_to_skip and any(
            skip_dir in root for skip_dir in directories_to_skip
        ):
            continue
        for file in files:
            if directories_to_skip and any(
                skip_dir in root for skip_dir in directories_to_skip
            ):
                continue
            if file.endswith(".xdf"):
                xdf_files.append(os.path.join(root, file))

    if not xdf_files:
        logging.warning(f"No xdf files found in {visit_dir}")

    return xdf_files


def is_already_done_panu_in_visit(visitPath, checkLog_fname):
    """
    Check if the visit was already processed with panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)
    return os.path.isfile(full_checkLog_fname)


def create_panu_log_file(visitPath, checkLog_fname):
    """
    Create the log file for panu
    """
    absVisitPath = os.path.abspath(visitPath)
    full_checkLog_fname = os.path.join(absVisitPath, checkLog_fname)

    # Create the log file
    logging.basicConfig(
        filename=full_checkLog_fname,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        force=True,  # remove previous handlers and set the new one
    )

    return full_checkLog_fname


def merge_panu_png_files_to_pdf(visit_path):
    """
    Merge the panu png files in the visit folder
    """
    png_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.png")]
    png_files.sort()

    if len(png_files) > 1:
        # read the png files
        images = [
            Image.open(os.path.join(visit_path, png_file)) for png_file in png_files
        ]
        # convert to RGB
        images = [img.convert("RGB") for img in images]

        images[0].save(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_png.pdf"),
            save_all=True,
            append_images=images[1:],
        )

        # remove the original png files
        for png_file in png_files:
            os.remove(os.path.join(visit_path, png_file))
            # print(f"    Removed {png_file}")
    else:
        msg = "No panu png files to merge"
        logging.info(msg)
        print(msg)


def merge_panu_pdf_files_to_pdf(visit_path):
    """
    Merge the panu pdf files in the visit folder
    """
    pdf_files = [f for f in os.listdir(visit_path) if f.endswith("_panu.pdf")]
    pdf_files.sort()

    from pypdf import PdfWriter

    if len(pdf_files) > 1:
        # read the pdf files
        pdf_merger = PdfWriter()
        for pdf_file in pdf_files:
            pdf_merger.append(os.path.join(visit_path, pdf_file))

        pdf_merger.write(
            os.path.join(visit_path, f"{os.path.basename(visit_path)}_panu_pdf.pdf")
        )
        pdf_merger.close()

        # remove the original pdf files
        for pdf_file in pdf_files:
            os.remove(os.path.join(visit_path, pdf_file))
            # print(f"    Removed {pdf_file}")
            pass
    else:
        msg = "No panu pdf files to merge"
        logging.info(msg)
        print(msg)


def get_panu_in_visit(visit_dir, directories_to_skip=None):
    """Correct the kinect timestamps for all the xdf files in the visit_dir"""

    panu_log = "panu.log"
    xdf_files = get_xdf_files_in_visit(visit_dir, directories_to_skip)

    if not xdf_files or len(xdf_files) == 0:
        return

    if is_already_done_panu_in_visit(visit_dir, panu_log):
        print(f"    Already done: '{panu_log}' found")
        return

    create_panu_log_file(visit_dir, panu_log)
    logging.info(f"Starting panu in {visit_dir}")

    for xdf_fullFname in xdf_files:
        if is_reach_file(xdf_fullFname):
            print(f"---- \n{xdf_fullFname}")
            logging.info(f"{os.path.basename(xdf_fullFname)}")
            save_panu(xdf_fullFname)

    merge_panu_png_files_to_pdf(os.path.dirname(visit_dir))
    logging.info("panu completed")
    print(f"    panu completed: see '{panu_log}' for details")


if doRunTests:
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"
    visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_dir = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"

    visit_dir = "../dat/ReArm.lnk/C1P42/V1"
    visit_dir = "../dat/ReArm.lnk/C1P42/V2"
    visit_dir = "../dat/ReArm.lnk/C1P42/V3"

    visit_dir = (
        "../dat/ReArm.lnk/DATA_named/C1P02/V2"  # /Armeo/002_CorJea_20210409_2_a.xdf"
    )

    visit_dir = "../dat/ReArm.lnk/DATA_named/C1P01/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P21/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P23/V1"
    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P20/V2"

    # visit_dir = "../dat/ReArm.lnk/DATA_named/C1P38/V1"

    os.remove(os.path.join(visit_dir, "panu.log"))
    get_panu_in_visit(visit_dir, directories_to_skip=["old", "Training", "Armeo"])